# Validación del modelo

Una vez que disponemos de un modelo entrenado (vía _Supervised Learning_ o _Reinforcement Learning_), procedemos a evaluar su desempeño utilizando instancias de **benchmark**.

Primero, cargaremos los pesos del modelo y configuraremos los adaptadores que definen cómo se procesará la entrada durante la inferencia.

In [1]:
from models.actions.cpmp_transformer_V1 import CPMPTransformer
from training.training import load_model
from generation.adapters.input import EnrichedLayoutAdapter, Layout4DAdapterV1
from generation.adapters.input.stack_features import StackFeaturesAdapterV1

# Cargamos la arquitectura y los pesos del modelo
model_name = "best_sl"
model = load_model(CPMPTransformer, model_name)

# Configuramos el adaptador de entrada (debe coincidir con el usado en el entrenamiento)
input_adapter = EnrichedLayoutAdapter(Layout4DAdapterV1, StackFeaturesAdapterV1)

## Evaluación Greedy

Para resolver las instancias, el modelo no actúa solo; se integra dentro de un **solver** que define la estrategia de toma de decisiones. La variante más directa es la **Greedy**, donde el modelo elige en cada paso la acción con la probabilidad más alta.

### Evaluación de una instancia

Utilizaremos la función `solve` para testear el modelo sobre un archivo específico. Esta nos devuelve el estado de la resolución (`solved`), el número de movimientos (`steps`) y el tiempo de cómputo invertido.

In [5]:
from solvers.model import ModelSolver

# Configuración de la prueba
instance_file = "benchmarks/5-5/data5-5-1.dat"
max_steps = 50
H = 7

# Inicializamos el solver con nuestro modelo y adaptador
solver = ModelSolver(model, input_adapter)

# Ejecutamos la resolución de la instancia
solved, steps, exec_time = solver.solve(instance_file, H, max_steps)

print(f"¿Resuelto?: {solved}")
print(f"Pasos realizados: {steps}")
print(f"Tiempo de ejecución: {exec_time:.4f}s")

¿Resuelto?: True
Pasos realizados: 30
Tiempo de ejecución: 0.0548s


### Evaluación por lotes

En lugar de probar instancias una a una, también podemos procesar todos los archivos de una carpeta específica.

In [12]:
from solvers.model import ModelSolver
from solvers.utils import summary

# Definimos la ruta del benchmark y parámetros del layout
folder = "benchmarks/5-5"
max_steps = 50
H = 7

# Inicializamos el solver
solver = ModelSolver(model, input_adapter)

# Resolvemos todas las instancias presentes en la carpeta
results = solver.solve_from_folder(folder, H, max_steps)

# Visualizamos el resumen estadístico (instancias resueltas y promedio de pasos)
summary(results)

Instancias resueltas: 40/40
Promedio de pasos: 32.70


## Beam Search (BSG)

Además de la variante Greedy, el modelo se puede integrar en marcos de optimización más robustos basados en **Beam Search**. Estas estrategias exploran múltiples soluciones en paralelo, utilizando el modelo para guiar la búsqueda.

### BSG-Full

En esta variante, expandimos los nodos en cada nivel (según un ancho de haz $w$) y ejecutamos _greedy rollouts_ con el propio modelo para evaluar qué tan prometedor es cada estado.

In [3]:
from solvers.bsg import BSGModelSolver

# w=2 indica que mantendremos los 2 mejores caminos en cada paso
solver = BSGModelSolver(model, input_adapter, w=2)
results = solver.solve_from_folder("benchmarks/5-5", H=7, max_steps=50)
summary(results)

Instancias resueltas: 40/40
Promedio de pasos: 28.15


### BSG-CostPredictor

Esta versión optimiza el tiempo de evaluación. En lugar de hacer _rollouts_ largos, empleamos un segundo modelo (un **Cost Predictor**) que estima directamente cuántos pasos faltan para resolver el _layout_.

In [4]:
from solvers.bsg import BSGCostPredictorSolver
from models.cost.cost_predictor import CostPredictorTransformer

# Cargamos el modelo especializado en predecir costos
cost_model = load_model(CostPredictorTransformer, "cost")

solver = BSGCostPredictorSolver(model, cost_model, input_adapter, w=2)
results = solver.solve_from_folder("benchmarks/5-5", H=7, max_steps=50)
summary(results)

Instancias resueltas: 40/40
Promedio de pasos: 28.65


### BSG-Hybrid

Combina la potencia de búsqueda del modelo de acciones con la rapidez de una heurística. Aquí, los estados se evalúan mediante _rollouts_ rápidos usando **FRG**.

In [5]:
from solvers.bsg.bsg_hybrid import BSGHybridSolver

solver = BSGHybridSolver(model, input_adapter, w=2)
results = solver.solve_from_folder("benchmarks/5-5", H=7, max_steps=50)
summary(results)

Instancias resueltas: 40/40
Promedio de pasos: 30.02


## Doble esfuerzo

El mecanismo **Double Search Effort** es una estrategia envolvente que permite escalar la intensidad de la búsqueda de forma dinámica. El proceso funciona por ciclos:

1.  Comenzamos con un valor inicial de ancho de haz ($w$).
2.  Ejecutamos el solver y, al finalizar, incrementamos $w$ multiplicándolo por un factor de $\sqrt{2}$.
3.  Repetimos el ciclo hasta agotar un tiempo máximo predefinido (`max_time`).

Este enfoque es excelente para evaluar el desempeño del modelo bajo presión temporal y para asegurar que, si existe una solución mejor que requiere más exploración, el solver eventualmente la encuentre.

In [10]:
from solvers.double_effort import DoubleEffortSolver
from solvers.bsg import BSGModelSolver

# Configuración de la prueba
instance_file = "benchmarks/5-5/data5-5-1.dat"
max_steps = 50
H = 7
max_time = 10 # Tiempo límite en segundos

# Definimos el solver base (por ejemplo, BSGModelSolver con w inicial de 1)
base_solver = BSGModelSolver(model, input_adapter, w=1)

# Envolvemos el base_solver con la lógica de Double Effort
dse_solver = DoubleEffortSolver(base_solver, max_time)

# Ejecutamos la búsqueda escalonada
solved, steps = dse_solver.solve(instance_file, H, max_steps)

print(f"¿Resuelto?: {solved} | Pasos finales: {steps}")

¿Resuelto?: True | Pasos finales: 30


## Evaluadores

Para automatizar la comparación, contamos con funciones de evaluación que ejecutan múltiples solvers sobre un mismo **benchmark** y consolidan los resultados.

Antes de comenzar, nos aseguramos de tener cargados tanto el modelo de acciones como el de predicción de costos, utilizando el adaptador de entrada correspondiente.

In [ ]:
from models.actions.cpmp_transformer_V1 import CPMPTransformer
from models.cost.cost_predictor import CostPredictorTransformer
from training.training import load_model
from generation.adapters.input import EnrichedLayoutAdapter, Layout4DAdapterV1
from generation.adapters.input.stack_features import StackFeaturesAdapterV1

# Cargamos la arquitectura y los pesos de los modelos
action_model = load_model(CPMPTransformer, "best_rl")
cost_model = load_model(CostPredictorTransformer, "cost")

# Configuramos el adaptador de entrada (debe coincidir con el usado en el entrenamiento)
input_adapter = EnrichedLayoutAdapter(Layout4DAdapterV1, StackFeaturesAdapterV1)

### Comparación de Beam Search

La función `solver_eval` permite contrastar distintas variantes de búsqueda (Heurística pura vs. Modelos de ML) bajo un mismo ancho de haz ($w$). Esto facilita identificar qué combinación de modelo y estrategia ofrece el mejor balance de pasos y tasa de éxito.

In [ ]:
from solvers import BSGFRGSolver, BSGModelSolver, BSGHybridSolver, BSGCostPredictorSolver
from evaluators.evaluator import solver_eval

# Definimos el benchmark y los parámetros comunes
folder = "benchmarks/5-5"
H, max_steps, w = 7, 50, 2

# Lista de solvers que deseamos contrastar
solvers = [
    BSGFRGSolver(w), # Heurística clásica
    BSGModelSolver(action_model, input_adapter, w),
    BSGHybridSolver(action_model, input_adapter, w),
    BSGCostPredictorSolver(action_model, cost_model, input_adapter, w)
]

# Ejecutamos la comparativa automática
solver_eval(solvers, folder, H, max_steps)

Solver                         | Instancia  | Avg Solved   | Avg Steps    | Avg Time    
----------------------------------------------------------------------------------------
BSGFRGSolver                   | 40         | 1.0000       | 28.9500      | 0.0177      
BSGModelSolver                 | 40         | 1.0000       | 28.1500      | 1.0860      
BSGHybridSolver                | 40         | 1.0000       | 30.0250      | 0.2164      
BSGCostPredictorSolver         | 40         | 1.0000       | 28.6500      | 0.1090      


,instance,Solved BSGFRGSolver,Steps BSGFRGSolver,Time BSGFRGSolver,Solved BSGModelSolver,Steps BSGModelSolver,Time BSGModelSolver,Solved BSGHybridSolver,Steps BSGHybridSolver,Time BSGHybridSolver,Solved BSGCostPredictorSolver,Steps BSGCostPredictorSolver,Time BSGCostPredictorSolver
0,data5-5-28.dat,True,33,0.020679,True,34,1.643499,True,31,0.228980,True,32,0.125973
1,data5-5-7.dat,True,30,0.020764,True,26,0.857463,True,29,0.210098,True,28,0.107968
2,data5-5-2.dat,True,33,0.027851,True,29,1.240655,True,36,0.259514,True,31,0.121095
3,data5-5-37.dat,True,26,0.015720,True,24,0.796088,True,25,0.178355,True,25,0.094099
4,data5-5-3.dat,True,28,0.019202,True,27,1.026144,True,28,0.207035,True,30,0.115118
5,data5-5-32.dat,True,30,0.016146,True,29,1.086445,True,28,0.203510,True,27,0.105636
6,data5-5-16.dat,True,30,0.024375,True,29,1.231515,True,27,0.200711,True,28,0.115523
7,data5-5-18.dat,True,27,0.021126,True,28,1.039911,True,36,0.276519,True,29,0.119000
8,data5-5-38.dat,True,25,0.011339,True,24,0.790294,True,26,0.189176,True,24,0.099074
9,data5-5-19.dat,True,31,0.018899,True,28,1.080887,True,30,0.228945,True,28,0.118426


### Evaluación con Double Search Effort

Si lo que buscamos es comparar el potencial máximo de los modelos bajo una restricción de tiempo, emplearemos `dse_eval`. Esta función aplica el mecanismo de doble esfuerzo de búsqueda a cada solver de la lista, permitiendo observar cómo evoluciona su precisión a medida que disponen de más tiempo para ejecutarse.

In [15]:
from solvers import BSGFRGSolver, BSGModelSolver, BSGHybridSolver, BSGCostPredictorSolver
from evaluators.dse_evaluator import dse_eval

# Configuramos las instancias y solvers a evaluar
folder = "benchmarks/5-5"
H = 7
max_steps = 50
w = 1
max_time = 5

solvers = [
    BSGFRGSolver(w),
    BSGModelSolver(model, input_adapter, w),
    BSGHybridSolver(model, input_adapter, w),
    BSGCostPredictorSolver(action_model, cost_model, input_adapter, w)
]

dse_eval(solvers, folder, H, max_steps, max_time)

Solver                         | Instancia  | Avg Solved   | Avg Steps   
-------------------------------------------------------------------------
DSE BSGFRGSolver               | 40         | 1.0000       | 26.9000     
DSE BSGModelSolver             | 40         | 1.0000       | 27.1000     
DSE BSGHybridSolver            | 40         | 1.0000       | 27.4000     
DSE BSGCostPredictorSolver     | 40         | 1.0000       | 26.0000     


,instance,Solved DSE BSGFRGSolver,Steps DSE BSGFRGSolver,Solved DSE BSGModelSolver,Steps DSE BSGModelSolver,Solved DSE BSGHybridSolver,Steps DSE BSGHybridSolver,Solved DSE BSGCostPredictorSolver,Steps DSE BSGCostPredictorSolver
0,data5-5-28.dat,True,30,True,34,True,31,True,28
1,data5-5-7.dat,True,26,True,26,True,26,True,25
2,data5-5-2.dat,True,31,True,29,True,33,True,30
3,data5-5-37.dat,True,24,True,24,True,25,True,22
4,data5-5-3.dat,True,25,True,26,True,25,True,26
5,data5-5-32.dat,True,27,True,26,True,28,True,26
6,data5-5-16.dat,True,25,True,26,True,25,True,25
7,data5-5-18.dat,True,27,True,26,True,27,True,26
8,data5-5-38.dat,True,22,True,23,True,24,True,23
9,data5-5-19.dat,True,29,True,28,True,29,True,26
